In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

In [ ]:
# =========
# Load data
# =========
df = pd.read_excel("your_user_data.xlsx")

In [3]:

# =========
# Reverse-coded items (inferred from internal consistency)
# 7-point reverse: reversed = 8 - x
# =========
df["post20_r"] = 8 - df["post20"]         # CI reverse
df["post11_r"] = 8 - df["post11"]         # Trust reverse
df["post13_r"] = 8 - df["post13"]         # Trust reverse
df["post14_r"] = 8 - df["post14"]         # Trust reverse

# =========
# Construct scales (use mean scores)
# =========
threat_items = [f"post{i}" for i in range(1, 7)]                 # post1-6
ui_items     = [f"post{i}" for i in range(7, 10)]                # post7-9
trust_items  = ["post10","post11_r","post12","post13_r","post14_r","post15","post16","post17"]
ci_items     = ["post18","post19","post20_r"]                    # post20 reversed

df["perceived_threat"]      = df[threat_items].mean(axis=1)
df["ui_satisfaction"]       = df[ui_items].mean(axis=1)
df["trust"]                 = df[trust_items].mean(axis=1)
df["continuance_intention"] = df[ci_items].mean(axis=1)

# =========
# Coding predictors
# =========
# bot_assignment: A=0, B=1
df["bot_B"] = (df["bot_assignment"] == "B").astype(int)

# Mean-center pp_score
df["pp_c"] = df["pp_score"] - df["pp_score"].mean()

# =========
# Cronbach alpha helper
# =========
def cronbach_alpha(items_df: pd.DataFrame) -> float:
    x = items_df.dropna()
    k = x.shape[1]
    item_vars = x.var(axis=0, ddof=1)
    total_var = x.sum(axis=1).var(ddof=1)
    return (k/(k-1)) * (1 - item_vars.sum()/total_var)

print("Alpha - Threat:", cronbach_alpha(df[threat_items]))
print("Alpha - UI:", cronbach_alpha(df[ui_items]))
print("Alpha - Trust (w/ reverse):", cronbach_alpha(df[trust_items]))
print("Alpha - CI (w/ reverse):", cronbach_alpha(df[ci_items]))

# =========
# Moderation models (OLS) : DV ~ bot + pp + bot*pp
# =========
def run_model(dv: str):
    formula = f"{dv} ~ bot_B + pp_c + bot_B:pp_c"
    m = smf.ols(formula, data=df).fit()
    print(f"\n=== {dv} ===")
    print("bot_B        :", m.params["bot_B"], "p=", m.pvalues["bot_B"])
    print("pp_c         :", m.params["pp_c"], "p=", m.pvalues["pp_c"])
    print("bot_B:pp_c   :", m.params["bot_B:pp_c"], "p=", m.pvalues["bot_B:pp_c"])
    print("R2           :", m.rsquared)
    return m

m_rq1 = run_model("continuance_intention")
m_threat = run_model("perceived_threat")
m_ui = run_model("ui_satisfaction")
m_trust = run_model("trust")

Alpha - Threat: 0.9380087404983326
Alpha - UI: 0.9246990385392261
Alpha - Trust (w/ reverse): 0.782933128057411
Alpha - CI (w/ reverse): 0.9290054005400541

=== continuance_intention ===
bot_B        : 2.167949559964672 p= 7.54597374382385e-06
pp_c         : 0.020114942528735913 p= 0.6892536565849092
bot_B:pp_c   : 0.020087990005604357 p= 0.7509717158656558
R2           : 0.3484629257029713

=== perceived_threat ===
bot_B        : -0.7259438265590477 p= 0.06255234600098941
pp_c         : 0.08333333333333354 p= 0.0647077671362426
bot_B:pp_c   : -0.06106431021755248 p= 0.27651146171474916
R2           : 0.16122188366625112

=== ui_satisfaction ===
bot_B        : 1.4435959295316279 p= 1.2766467893965625e-05
pp_c         : 0.01652298850574737 p= 0.632730296807008
bot_B:pp_c   : 0.017238790121200907 p= 0.6919086150698481
R2           : 0.3373896725343978

=== trust ===
bot_B        : 1.3054522457717448 p= 5.835288519782195e-07
pp_c         : 0.03798491379310376 p= 0.15424087073506065
bot_B: